# 01. Data Integration and Dataset Understanding

In this notebook, we will understand and organize the MoA Prediction dataset before doing EDA, feature engineering, or model training.

The main goals of this notebook are:

- load all raw dataset files,
- check the shape of each file,
- understand the role of each table,
- identify important columns,
- check duplicate `sig_id` values,
- check missing values,
- check whether IDs match across files,
- create a safe data integration plan.

At this stage, we will not train any model.  
We will only prepare the dataset correctly.

In [30]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

In [31]:
# Detect project root safely
PROJECT_ROOT = Path.cwd()

# If notebook is running from the notebooks folder, go one level up
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DATA_DIR = PROJECT_ROOT / "data" / "interim"

INTERIM_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data folder:", RAW_DATA_DIR)
print("Interim data folder:", INTERIM_DATA_DIR)

Project root: c:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Machine_Learning\moa-prediction-drug-response
Raw data folder: c:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Machine_Learning\moa-prediction-drug-response\data\raw
Interim data folder: c:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Machine_Learning\moa-prediction-drug-response\data\interim


## 1. Project Setup and Raw Data Loading

This section sets up the notebook, checks whether the required raw files are available, loads the CSV files, and creates the first shape overview.

### 1.1 Check Required Raw Files

Before loading the dataset, we first check whether all required CSV files are available inside the `data/raw/` folder.

This prevents errors later and confirms that the project folder is correctly organized.

In [32]:
required_files = [
    "train_features.csv",
    "test_features.csv",
    "train_targets_scored.csv",
    "train_targets_nonscored.csv",
    "train_drug.csv",
    "sample_submission.csv",
]

file_check = []

for file_name in required_files:
    file_path = RAW_DATA_DIR / file_name
    file_check.append({
        "file_name": file_name,
        "exists": file_path.exists(),
        "path": str(file_path)
    })

file_check_df = pd.DataFrame(file_check)
file_check_df

,file_name,exists,path
0,train_features.csv,True,c:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...
1,test_features.csv,True,c:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...
2,train_targets_scored.csv,True,c:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...
3,train_targets_nonscored.csv,True,c:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...
4,train_drug.csv,True,c:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...
5,sample_submission.csv,True,c:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...


### 1.2 Load Raw Dataset Files

Now we load all raw CSV files into pandas DataFrames.

At this stage, we only load the data.  
We will not merge anything yet because we first need to check shapes, columns, duplicate IDs, and ID alignment.

In [33]:
train_features = pd.read_csv(RAW_DATA_DIR / "train_features.csv")
test_features = pd.read_csv(RAW_DATA_DIR / "test_features.csv")

train_targets_scored = pd.read_csv(RAW_DATA_DIR / "train_targets_scored.csv")
train_targets_nonscored = pd.read_csv(RAW_DATA_DIR / "train_targets_nonscored.csv")

train_drug = pd.read_csv(RAW_DATA_DIR / "train_drug.csv")
sample_submission = pd.read_csv(RAW_DATA_DIR / "sample_submission.csv")

print("All files loaded successfully.")

All files loaded successfully.


### 1.3 First Look at Dataset Shapes

After loading the files, we check the number of rows and columns in each table.

This helps us understand which files contain input features, which files contain targets, and whether the row counts match as expected.

In [34]:
file_overview = pd.DataFrame({
    "file_name": [
        "train_features",
        "test_features",
        "train_targets_scored",
        "train_targets_nonscored",
        "train_drug",
        "sample_submission",
    ],
    "rows": [
        train_features.shape[0],
        test_features.shape[0],
        train_targets_scored.shape[0],
        train_targets_nonscored.shape[0],
        train_drug.shape[0],
        sample_submission.shape[0],
    ],
    "columns": [
        train_features.shape[1],
        test_features.shape[1],
        train_targets_scored.shape[1],
        train_targets_nonscored.shape[1],
        train_drug.shape[1],
        sample_submission.shape[1],
    ],
})

file_overview

,file_name,rows,columns
0,train_features,23814,876
1,test_features,3982,876
2,train_targets_scored,23814,207
3,train_targets_nonscored,23814,403
4,train_drug,23814,2
5,sample_submission,3982,207


### 1.4 Dataset File Roles

The MoA dataset is divided into input feature files, target files, drug information, and submission format.

- `train_features.csv` is the main training input table. It contains metadata, gene expression features, and cell viability features.
- `test_features.csv` is the main test input table. It has the same feature columns as `train_features.csv`, but it does not contain target labels.
- `train_targets_scored.csv` contains the main target labels used for model training and Kaggle scoring.
- `train_targets_nonscored.csv` contains extra target labels. These are not required for baseline modeling, but they may be useful later for auxiliary learning.
- `train_drug.csv` contains anonymous drug IDs for training samples only.
- `sample_submission.csv` shows the required final submission format.

At this stage, we only understand the files. We will not merge anything yet.

## 2. Column and Feature Group Understanding

This section identifies the important column groups and checks whether the train and test feature columns are consistent.

### 2.1 Identify Column Groups

Now we separate the columns into meaningful groups.

The main column groups are:

- ID column: `sig_id`
- metadata columns: `cp_type`, `cp_time`, `cp_dose`
- gene expression columns: columns starting with `g-`
- cell viability columns: columns starting with `c-`
- scored target columns: target labels from `train_targets_scored.csv`
- nonscored target columns: auxiliary labels from `train_targets_nonscored.csv`

This step is important because later we will use different feature groups for EDA, feature engineering, and model training.

In [35]:
ID_COL = "sig_id"

metadata_features = ["cp_type", "cp_time", "cp_dose"]

gene_features = [col for col in train_features.columns if col.startswith("g-")]
cell_features = [col for col in train_features.columns if col.startswith("c-")]

scored_target_features = [
    col for col in train_targets_scored.columns 
    if col != ID_COL
]

nonscored_target_features = [
    col for col in train_targets_nonscored.columns 
    if col != ID_COL
]

feature_group_summary = pd.DataFrame({
    "feature_group": [
        "ID column",
        "metadata features",
        "gene expression features",
        "cell viability features",
        "scored target labels",
        "nonscored target labels",
    ],
    "count": [
        1,
        len(metadata_features),
        len(gene_features),
        len(cell_features),
        len(scored_target_features),
        len(nonscored_target_features),
    ]
})

feature_group_summary

,feature_group,count
0,ID column,1
1,metadata features,3
2,gene expression features,772
3,cell viability features,100
4,scored target labels,206
5,nonscored target labels,402


### 2.2 Check Important Column Names

Before doing validation or merging, we quickly check whether the expected key and metadata columns exist.

This helps prevent future errors during merging and feature engineering.

In [36]:
important_column_check = {
    "sig_id_in_train_features": ID_COL in train_features.columns,
    "sig_id_in_test_features": ID_COL in test_features.columns,
    "sig_id_in_train_targets_scored": ID_COL in train_targets_scored.columns,
    "sig_id_in_train_targets_nonscored": ID_COL in train_targets_nonscored.columns,
    "sig_id_in_train_drug": ID_COL in train_drug.columns,
    "sig_id_in_sample_submission": ID_COL in sample_submission.columns,
    "metadata_columns_exist": all(col in train_features.columns for col in metadata_features),
    "drug_id_exists": "drug_id" in train_drug.columns,
}

important_column_check

{'sig_id_in_train_features': True,
 'sig_id_in_test_features': True,
 'sig_id_in_train_targets_scored': True,
 'sig_id_in_train_targets_nonscored': True,
 'sig_id_in_train_drug': True,
 'sig_id_in_sample_submission': True,
 'metadata_columns_exist': True,
 'drug_id_exists': True}

### 2.3 Check Train and Test Feature Consistency

The training and test feature tables should contain the same input feature columns.

This is important because the model will be trained on `train_features.csv` and later used to predict on `test_features.csv`.

If train and test feature columns do not match, model training or prediction will fail.

In [37]:
train_feature_cols = set(train_features.columns) - {ID_COL}
test_feature_cols = set(test_features.columns) - {ID_COL}

missing_in_test = train_feature_cols - test_feature_cols
extra_in_test = test_feature_cols - train_feature_cols

column_consistency_report = {
    "train_feature_count_without_id": len(train_feature_cols),
    "test_feature_count_without_id": len(test_feature_cols),
    "missing_in_test": list(missing_in_test),
    "extra_in_test": list(extra_in_test),
    "columns_match": len(missing_in_test) == 0 and len(extra_in_test) == 0,
}

column_consistency_report

{'train_feature_count_without_id': 875,
 'test_feature_count_without_id': 875,
 'missing_in_test': [],
 'extra_in_test': [],
 'columns_match': True}

## 3. Data Quality and ID Validation

This section checks duplicate IDs, missing values, ID matching, row alignment, missing or extra IDs, and target validity before any safe merge.

### 3.1 Check Duplicate `sig_id` Values

The `sig_id` column is the unique identifier for each sample.

Before merging any files, we must confirm that each `sig_id` appears only once in every table. If duplicate IDs exist, merging can create duplicated rows and damage the dataset structure.

This check helps us make sure that the data can be safely connected using `sig_id`.

In [38]:
# Store all loaded DataFrames in one dictionary for repeated checks

files = {
    "train_features": train_features,
    "test_features": test_features,
    "train_targets_scored": train_targets_scored,
    "train_targets_nonscored": train_targets_nonscored,
    "train_drug": train_drug,
    "sample_submission": sample_submission,
}

files.keys()

dict_keys(['train_features', 'test_features', 'train_targets_scored', 'train_targets_nonscored', 'train_drug', 'sample_submission'])

In [39]:
duplicate_id_report = []

for name, df in files.items():
    duplicate_id_report.append({
        "file_name": name,
        "total_rows": df.shape[0],
        "unique_sig_id": df[ID_COL].nunique(),
        "duplicate_sig_id_count": df[ID_COL].duplicated().sum(),
        "is_sig_id_unique": df[ID_COL].is_unique,
    })

duplicate_id_report = pd.DataFrame(duplicate_id_report)
duplicate_id_report

,file_name,total_rows,unique_sig_id,duplicate_sig_id_count,is_sig_id_unique
0,train_features,23814,23814,0,True
1,test_features,3982,3982,0,True
2,train_targets_scored,23814,23814,0,True
3,train_targets_nonscored,23814,23814,0,True
4,train_drug,23814,23814,0,True
5,sample_submission,3982,3982,0,True


### 3.2 Check Missing Values

Now we check whether any file contains missing values.

Missing values can cause problems during feature engineering and model training. Even if we expect this dataset to be clean, we should always check it before moving forward.

In [40]:
missing_value_report = []

for name, df in files.items():
    total_missing = df.isna().sum().sum()
    columns_with_missing = (df.isna().sum() > 0).sum()
    
    missing_value_report.append({
        "file_name": name,
        "total_missing_values": total_missing,
        "columns_with_missing": columns_with_missing,
        "missing_percentage": round((total_missing / df.size) * 100, 4),
    })

missing_value_report = pd.DataFrame(missing_value_report)
missing_value_report

,file_name,total_missing_values,columns_with_missing,missing_percentage
0,train_features,0,0,0.0
1,test_features,0,0,0.0
2,train_targets_scored,0,0,0.0
3,train_targets_nonscored,0,0,0.0
4,train_drug,0,0,0.0
5,sample_submission,0,0,0.0


In [41]:
for name, df in files.items():
    missing_by_column = df.isna().sum()
    missing_by_column = missing_by_column[missing_by_column > 0].sort_values(ascending=False)
    
    if len(missing_by_column) > 0:
        print(f"\n{name} columns with missing values:")
        display(missing_by_column)

### 3.3 Check ID Matching Across Files

Before merging tables, we need to confirm that the same `sig_id` values exist across related files.

For the training data, `train_features`, `train_targets_scored`, `train_targets_nonscored`, and `train_drug` should contain the same training sample IDs.

For the test data, `test_features` and `sample_submission` should contain the same test sample IDs.

This check confirms whether the files can be safely connected using `sig_id`.

In [42]:
id_matching_report = pd.DataFrame({
    "check": [
        "train_features vs train_targets_scored",
        "train_features vs train_targets_nonscored",
        "train_features vs train_drug",
        "test_features vs sample_submission",
    ],
    "same_id_set": [
        set(train_features[ID_COL]) == set(train_targets_scored[ID_COL]),
        set(train_features[ID_COL]) == set(train_targets_nonscored[ID_COL]),
        set(train_features[ID_COL]) == set(train_drug[ID_COL]),
        set(test_features[ID_COL]) == set(sample_submission[ID_COL]),
    ],
})

id_matching_report

,check,same_id_set
0,train_features vs train_targets_scored,True
1,train_features vs train_targets_nonscored,True
2,train_features vs train_drug,True
3,test_features vs sample_submission,True


### 3.4 Check Row Alignment

After checking that the same IDs exist across files, we also check whether they are in the same row order.

This is important because some people directly join or compare tables by row position. However, row order should never be trusted blindly.

Even if row order is different, we can still safely merge using `sig_id`.

In [43]:
row_alignment_report = pd.DataFrame({
    "check": [
        "train_features vs train_targets_scored",
        "train_features vs train_targets_nonscored",
        "train_features vs train_drug",
        "test_features vs sample_submission",
    ],
    "same_row_order": [
        (train_features[ID_COL].values == train_targets_scored[ID_COL].values).all(),
        (train_features[ID_COL].values == train_targets_nonscored[ID_COL].values).all(),
        (train_features[ID_COL].values == train_drug[ID_COL].values).all(),
        (test_features[ID_COL].values == sample_submission[ID_COL].values).all(),
    ],
})

row_alignment_report

,check,same_row_order
0,train_features vs train_targets_scored,True
1,train_features vs train_targets_nonscored,True
2,train_features vs train_drug,True
3,test_features vs sample_submission,True


### 3.5 Check Missing or Extra IDs

If any ID matching check fails, we need to know which IDs are missing or extra.

This step creates a small diagnostic function. If everything is already matched, all missing and extra counts should be zero.

In [44]:
def compare_id_sets_clean(left_df, right_df, left_name, right_name, id_col=ID_COL):
    left_ids = set(left_df[id_col])
    right_ids = set(right_df[id_col])
    
    return {
        "comparison": f"{left_name} vs {right_name}",
        "left_table": left_name,
        "right_table": right_name,
        "left_ids_not_in_right": len(left_ids - right_ids),
        "right_ids_not_in_left": len(right_ids - left_ids),
        "ids_match": left_ids == right_ids,
    }


missing_extra_id_report = pd.DataFrame([
    compare_id_sets_clean(train_features, train_targets_scored, "train_features", "train_targets_scored"),
    compare_id_sets_clean(train_features, train_targets_nonscored, "train_features", "train_targets_nonscored"),
    compare_id_sets_clean(train_features, train_drug, "train_features", "train_drug"),
    compare_id_sets_clean(test_features, sample_submission, "test_features", "sample_submission"),
])

missing_extra_id_report

,comparison,left_table,right_table,left_ids_not_in_right,right_ids_not_in_left,ids_match
0,train_features vs train_targets_scored,train_features,train_targets_scored,0,0,True
1,train_features vs train_targets_nonscored,train_features,train_targets_nonscored,0,0,True
2,train_features vs train_drug,train_features,train_drug,0,0,True
3,test_features vs sample_submission,test_features,sample_submission,0,0,True


### 3.6 Check Target Validity

The MoA target columns should contain only binary values.

For both scored and nonscored targets:

- `0` means the target is not active.
- `1` means the target is active.

Before modeling, we need to confirm that there are no unexpected values in the target files.

In [45]:
scored_unique_values = np.unique(train_targets_scored[scored_target_features].values)
nonscored_unique_values = np.unique(train_targets_nonscored[nonscored_target_features].values)

target_validity_report = {
    "scored_unique_values": scored_unique_values.tolist(),
    "nonscored_unique_values": nonscored_unique_values.tolist(),
    "scored_targets_are_binary": set(scored_unique_values).issubset({0, 1}),
    "nonscored_targets_are_binary": set(nonscored_unique_values).issubset({0, 1}),
}

target_validity_report

{'scored_unique_values': [0, 1],
 'nonscored_unique_values': [0, 1],
 'scored_targets_are_binary': True,
 'nonscored_targets_are_binary': True}

## 4. Metadata, Target, and Control Validation

This section validates the metadata values, target-submission column consistency, and the behavior of control samples.

### 4.1 Check Metadata Column Values

The metadata columns describe the treatment setup for each sample.

We check the unique values of:

- `cp_type`: treatment type
- `cp_time`: treatment duration
- `cp_dose`: treatment dose

This helps us confirm that the metadata columns contain expected categories before EDA and feature engineering.

In [46]:
metadata_value_report = {
    "cp_type_unique_values": sorted(train_features["cp_type"].unique().tolist()),
    "cp_time_unique_values": sorted(train_features["cp_time"].unique().tolist()),
    "cp_dose_unique_values": sorted(train_features["cp_dose"].unique().tolist()),
}

metadata_value_report

{'cp_type_unique_values': ['ctl_vehicle', 'trt_cp'],
 'cp_time_unique_values': [24, 48, 72],
 'cp_dose_unique_values': ['D1', 'D2']}

### 4.2 Check Metadata Distribution

Now we check how many samples exist in each metadata category.

This is not full EDA yet.  
It is only a basic dataset understanding step to see whether the treatment types, time points, and dose levels are distributed properly.

In [47]:
metadata_distribution = {
    "cp_type_distribution": train_features["cp_type"].value_counts().to_dict(),
    "cp_time_distribution": train_features["cp_time"].value_counts().sort_index().to_dict(),
    "cp_dose_distribution": train_features["cp_dose"].value_counts().to_dict(),
}

metadata_distribution

{'cp_type_distribution': {'trt_cp': 21948, 'ctl_vehicle': 1866},
 'cp_time_distribution': {24: 7772, 48: 8250, 72: 7792},
 'cp_dose_distribution': {'D1': 12147, 'D2': 11667}}

### 4.3 Check Train and Test Metadata Consistency

The model will be trained on `train_features` and later used on `test_features`.

So the metadata categories in train and test should be consistent.

Here we check whether `cp_type`, `cp_time`, and `cp_dose` contain the same types of values in both train and test data.

In [48]:
metadata_consistency_report = []

for col in metadata_features:
    train_values = sorted(train_features[col].unique().tolist())
    test_values = sorted(test_features[col].unique().tolist())
    
    metadata_consistency_report.append({
        "column": col,
        "train_unique_values": train_values,
        "test_unique_values": test_values,
        "same_values": set(train_values) == set(test_values),
        "values_in_train_not_test": list(set(train_values) - set(test_values)),
        "values_in_test_not_train": list(set(test_values) - set(train_values)),
    })

metadata_consistency_report = pd.DataFrame(metadata_consistency_report)
metadata_consistency_report

,column,train_unique_values,test_unique_values,same_values,values_in_train_not_test,values_in_test_not_train
0,cp_type,"[ctl_vehicle, trt_cp]","[ctl_vehicle, trt_cp]",True,[],[]
1,cp_time,"[24, 48, 72]","[24, 48, 72]",True,[],[]
2,cp_dose,"[D1, D2]","[D1, D2]",True,[],[]


### 4.4 Check Target and Submission Column Consistency

The final submission must predict the same target columns that exist in `train_targets_scored.csv`.

Here we check whether the scored target columns match the columns in `sample_submission.csv`.

This is important because the final prediction file must have exactly the same target columns and column order.

In [49]:
sample_submission_target_features = [
    col for col in sample_submission.columns 
    if col != ID_COL
]

target_submission_report = {
    "scored_target_count": len(scored_target_features),
    "submission_target_count": len(sample_submission_target_features),
    "missing_in_submission": list(set(scored_target_features) - set(sample_submission_target_features)),
    "extra_in_submission": list(set(sample_submission_target_features) - set(scored_target_features)),
    "same_target_set": set(scored_target_features) == set(sample_submission_target_features),
    "same_target_order": scored_target_features == sample_submission_target_features,
}

target_submission_report

{'scored_target_count': 206,
 'submission_target_count': 206,
 'missing_in_submission': [],
 'extra_in_submission': [],
 'same_target_set': True,
 'same_target_order': True}

### 4.5 Check Control Sample Target Behavior

Control samples have `cp_type == "ctl_vehicle"`.

These samples should not have active MoA targets because they are vehicle control samples, not treated compound samples.

This check is important because later, during final prediction, we can set all target probabilities to zero for test rows where `cp_type == "ctl_vehicle"`.

In [50]:
control_ids = train_features.loc[
    train_features["cp_type"] == "ctl_vehicle",
    ID_COL
]

control_targets = train_targets_scored.loc[
    train_targets_scored[ID_COL].isin(control_ids),
    scored_target_features
]

control_report = {
    "control_sample_count": len(control_ids),
    "control_target_rows": control_targets.shape[0],
    "control_positive_target_count": int(control_targets.sum().sum()),
    "all_control_targets_zero": int(control_targets.sum().sum()) == 0,
}

control_report

{'control_sample_count': 1866,
 'control_target_rows': 1866,
 'control_positive_target_count': 0,
 'all_control_targets_zero': True}

## 5. Drug Information Review

This section reviews `train_drug.csv` to understand repeated drugs and prepare for possible drug-aware validation later.

### 5.1 Check Drug Information

The `train_drug.csv` file contains anonymous drug IDs for the training samples.

This file is useful for understanding repeated drugs and for possible drug-aware validation later.

However, `drug_id` should not be used directly in the baseline model because it is only available for the training data.

In [51]:
drug_report = {
    "train_drug_rows": train_drug.shape[0],
    "train_drug_columns": train_drug.shape[1],
    "unique_sig_id": train_drug[ID_COL].nunique(),
    "unique_drug_id": train_drug["drug_id"].nunique(),
    "duplicate_sig_id_count": train_drug[ID_COL].duplicated().sum(),
    "missing_drug_id_count": train_drug["drug_id"].isna().sum(),
}

drug_report

{'train_drug_rows': 23814,
 'train_drug_columns': 2,
 'unique_sig_id': 23814,
 'unique_drug_id': 3289,
 'duplicate_sig_id_count': np.int64(0),
 'missing_drug_id_count': np.int64(0)}

### 5.2 Check Repeated Drug Frequency

Some drugs appear multiple times in the training data.

This is important because repeated drugs can affect validation. If the same drug appears in both training and validation folds, the validation score may become slightly optimistic.

For the baseline model, we will not use `drug_id` as a feature. Later, we may use it for drug-aware validation.

In [52]:
drug_frequency_summary = train_drug["drug_id"].value_counts().describe()

drug_frequency_summary

count    3289.000000
mean        7.240499
std        35.901370
min         1.000000
25%         6.000000
50%         6.000000
75%         6.000000
max      1866.000000
Name: count, dtype: float64

In [53]:
top_repeated_drugs = (
    train_drug["drug_id"]
    .value_counts()
    .head(10)
    .reset_index()
)

top_repeated_drugs.columns = ["drug_id", "sample_count"]

top_repeated_drugs

,drug_id,sample_count
0,cacb2b860,1866
1,87d714366,718
2,9f80f3f77,246
3,8b87a7a83,203
4,5628cb3ee,202
5,d08af5d4b,196
6,292ab2c28,194
7,d50f18348,186
8,d1b47f29d,178
9,67c879e79,19
